In [ ]:
#@markdown # **1) Download required Libraries**
# # Clone Real-ESRGAN and enter the Real-ESRGAN
from google.colab import drive, files
from IPython.display import clear_output
import os, shutil, subprocess
drive_mounted = False
temp_folder = 'tmp_frames'
result_folder = 'results'
!git clone https://github.com/xinntao/Real-ESRGAN.git
clear_output()
%cd Real-ESRGAN
# Set up the environment
!pip install basicsr facexlib gfpgan
clear_output()
!pip install -r requirements.txt
clear_output()
!python setup.py develop
# Download the pre-trained model
!wget https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P experiments/pretrained_models
%cd ..
clear_output()
print(f"✅ Done Installing")

In [ ]:
#@markdown # **2. Upload Video**
if True:
  upload_folder = 'upload'

  if os.path.isdir(upload_folder):
      shutil.rmtree(upload_folder)

  os.mkdir(upload_folder)

  # upload images
  uploaded = files.upload()
  file_name = next(iter(uploaded))
  input_path = f'/content/{upload_folder}/{file_name}'
  for filename in uploaded.keys():
    dst_path = os.path.join(upload_folder, filename)
    print(f'move {filename} to {dst_path}')
    shutil.move(filename, dst_path)
INPUT_DIR = "/content/upload"
os.makedirs(INPUT_DIR, exist_ok=True)
orig_vid = input_path
safe_vid = "init_vid.mp4"
shutil.copy(orig_vid, os.path.join(INPUT_DIR, safe_vid))
print(f"✅ Video saved to {INPUT_DIR}/{safe_vid}")
your_vid = os.path.join(INPUT_DIR, safe_vid)

In [ ]:
#@markdown # **3. Genrate Video**

# Step 1: Patch Real-ESRGAN for compatibility
!sed -i 's/torchvision.transforms.functional_tensor/torchvision.transforms.functional/' \
    /usr/local/lib/python3.11/dist-packages/basicsr/data/degradations.py
!pip install ffmpeg
# Step 2: Run video inference using multiprocessing
import os
%cd /content/Real-ESRGAN

temp_in = your_vid
temp_out = '/content/output_raw.mp4'

!python inference_realesrgan_video.py \
    -i "{temp_in}" -o "{temp_out}" \
    -n RealESRGAN_x2plus \
    --outscale 4 --face_enhance \
    --num_process_per_gpu 3

%cd ..

# Step 3: Remux with ffmpeg to fix container issues
remuxed = '/content/output_raw.mp4/download_out.mp4'
!ffmpeg -y -i "{temp_out}" -c copy "{remuxed}"

# Step 4: Check file size and existence
import os
size = os.path.getsize(remuxed) / (1024*1024)
print(f"✅ Remuxed output saved: {remuxed} ({size:.2f} MB)")
